# Capítulo 1 — NumPy I: el array, la vectorización y el fin de los bucles

**Serie Ciencia de Datos · Tomo 2 — Herramientas Python para Data Science**  
José Fernando Meza

Código del capítulo, extraído del libro. Repositorio: https://github.com/josefer76/serie-ciencia-de-datos

> Ejecute las celdas en orden: algunas dependen de variables definidas antes.

## Código del capítulo

### 1.1 Por qué existe NumPy: limitaciones de las listas nativas y el coste real de los bucles en Python

In [ ]:
import sys

# Un entero "simple" en Python nunca es solo 8 bytes de datos:
# arrastra cabecera de objeto (conteo de referencias, puntero a tipo, etc.)
print(sys.getsizeof(1))            # 28 bytes en CPython (build de 64 bits)

# Una lista de 4 enteros no almacena los valores: almacena punteros a ellos
print(sys.getsizeof([1, 2, 3, 4]))  # 88 bytes solo para el contenedor

### 1.2 El objeto ndarray: memoria contigua, tipo homogéneo (dtype) y por qué eso lo hace rápido

In [ ]:
import numpy as np

arreglo = np.array([1, 2, 3, 4], dtype=np.int64)

print(arreglo.dtype)     # int64      -> tipo único para todo el arreglo
print(arreglo.itemsize)  # 8          -> bytes que ocupa CADA elemento
print(arreglo.nbytes)
# 32         -> itemsize * cantidad de elementos (4 * 8)

### 1.3 Creación de arrays: `array()`, `zeros()`, `ones()`, `arange()`, `linspace()`, `full()`

In [ ]:
import numpy as np

# array(): convierte una estructura Python existente (lista, lista de listas)
# en ndarray
vector = np.array([1, 2, 3, 4, 5])
matriz = np.array([[1, 2, 3], [4, 5, 6]])   # lista de listas -> matriz 2x3

# zeros(): arreglo inicializado en cero; típico para acumuladores o buffers
ceros = np.zeros((2, 3))          # dtype por defecto: float64
ceros_enteros = np.zeros((2, 3), dtype=np.int32)   # se puede forzar el tipo

# ones(): arreglo inicializado en uno; útil como base para máscaras o pesos
# iniciales
unos = np.ones((3, 3))

# arange(): equivalente vectorizado de range(), admite paso fraccionario
secuencia = np.arange(0, 10, 2)   # array([0, 2, 4, 6, 8])

# linspace(): n puntos equiespaciados INCLUYENDO ambos extremos
# (arange con paso fraccionario acumula error de redondeo; linspace no)
dominio = np.linspace(0, 1, 5)    # array([0.  , 0.25, 0.5 , 0.75, 1.  ])

# full(): arreglo relleno con un valor constante arbitrario
relleno = np.full((2, 2), 7)

### 1.4 Atributos fundamentales: `shape`, `ndim`, `size`, `dtype`, `itemsize`

In [ ]:
import numpy as np

matriz = np.array([[1, 2, 3], [4, 5, 6]])

print(matriz.shape)
# (2, 3)  -> tupla con el tamaño de cada dimensión (filas, columnas)
print(matriz.ndim)      # 2       -> número de dimensiones (aquí, una matriz)
print(matriz.size)
# 6       -> cantidad total de elementos = producto de shape
print(matriz.dtype)     # int64   -> tipo por defecto (ver nota abajo)
print(matriz.itemsize)  # 8       -> bytes que ocupa un solo elemento
print(matriz.nbytes)    # 48      -> size * itemsize

### 1.5 Indexación y slicing en 1D y 2D: vistas frente a copias

#### Indexación y slicing básico

In [ ]:
import numpy as np

vector = np.arange(10)          # array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
print(vector[3])                # 3            -> un solo elemento
print(vector[2:5])              # [2 3 4]       -> slice: posiciones 2, 3, 4

matriz = np.arange(12).reshape(3, 4)
# array([[ 0,  1,  2,  3],
#        [ 4,  5,  6,  7],
#        [ 8,  9, 10, 11]])

print(matriz[1, 2])              # 6      -> fila 1, columna 2
print(matriz[1, :])              # [4 5 6 7]   -> fila 1 completa
print(matriz[:, 2])              # [ 2  6 10]  -> columna 2 completa
print(matriz[0:2, 1:3])          # sub-matriz: filas 0-1, columnas 1-2
# [[1 2]
#  [5 6]]

In [ ]:
columna = matriz[:, 2]
print(columna.shape, columna.ndim)
# (3,) 1   -> la dimensión de columnas desapareció

# Para conservar la segunda dimensión (obtener un verdadero vector columna),
# se indexa con un slice de longitud 1 en lugar de un entero:
columna_2d = matriz[:, 2:3]
print(columna_2d.shape, columna_2d.ndim)
# (3, 1) 2   -> la dimensión se conserva, con tamaño 1

#### La distinción que no admite atajos: vistas frente a copias

In [ ]:
import numpy as np

original = np.arange(10)
vista = original[2:5]      # slicing básico -> VISTA, no copia

vista[0] = 999             # modificamos la vista...
print(original)            # ...y el original queda alterado:
# [  0   1 999   3   4   5   6   7   8   9]

print(vista.base is original)
# True -> vista.base apunta al array que le dio origen

In [ ]:
import numpy as np

original = np.arange(10)
copia = original[[2, 3, 4]]   # fancy indexing (lista de índices) -> COPIA

copia[0] = 999                # modificamos la copia...
print(original)               # ...el original permanece intacto:
# [0 1 2 3 4 5 6 7 8 9]

print(copia.base)
# None -> no depende de memoria ajena, es independiente

In [ ]:
independiente = original[2:5].copy()   # ahora sí es una copia real
independiente[0] = -1                  # el original queda intacto

### 1.6 Vectorización: operaciones elemento a elemento sin bucles explícitos

In [ ]:
import numpy as np

precios = np.array([100.0, 250.0, 80.0, 430.0])
impuesto = 0.16

# Forma NO vectorizada (la que usaríamos con una lista de Python en el Libro
# 1):
precios_con_impuesto_bucle = []
for precio in precios:
    precios_con_impuesto_bucle.append(precio * (1 + impuesto))

# Forma vectorizada: la operación se expresa sobre el arreglo completo
precios_con_impuesto = precios * (1 + impuesto)
# array([116. , 290. ,  92.8, 498.8])

# Comparaciones también son vectorizadas y devuelven un array booleano
# (esto anticipa las máscaras del Capítulo 2, Sección 2.3)
caros = precios > 200
# array([False,  True, False,  True])

### 1.7 Benchmark riguroso: lista + bucle frente a array vectorizado

#### Metodología de medición

In [ ]:
import numpy as np

arreglo = np.arange(1_000_000)   # el mismo arreglo del Benchmark 1
%timeit arreglo * 2

*Salida en el libro:*

```text
624 µs ± 49.9 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)
```

In [ ]:
import timeit

tiempos = timeit.repeat(lambda: arreglo * 2, repeat=7, number=10)
tiempo_estimado = min(tiempos) / 10   # segundos por ejecución

#### Benchmark 1 — Duplicar un millón de elementos

In [ ]:
import numpy as np
import timeit

n = 1_000_000
lista = list(range(n))
arreglo = np.arange(n)

def doblar_con_bucle(datos):
    resultado = []
    for x in datos:
        resultado.append(x * 2)
    return resultado

def doblar_con_comprension(datos):
    return [x * 2 for x in datos]

tiempo_bucle = min(timeit.repeat(
    lambda: doblar_con_bucle(lista), repeat=7, number=3
)) / 3

tiempo_comprension = min(timeit.repeat(
    lambda: doblar_con_comprension(lista), repeat=7, number=3
)) / 3

tiempo_vectorizado = min(timeit.repeat(
    lambda: arreglo * 2, repeat=7, number=10
)) / 10

print(f"Bucle for explícito  : {tiempo_bucle * 1000:8.2f} ms")
print(f"Comprensión de lista : {tiempo_comprension * 1000:8.2f} ms")
print(f"Vectorizado (NumPy)  : {tiempo_vectorizado * 1000:8.2f} ms")
print(f"Vectorizado frente a bucle       : "
      f"{tiempo_bucle / tiempo_vectorizado:.0f} veces más rápido")
print(f"Vectorizado frente a comprensión : "
      f"{tiempo_comprension / tiempo_vectorizado:.0f} veces más rápido")

*Salida en el libro:*

```text
Bucle for explícito  :    32.43 ms
Comprensión de lista :    34.23 ms
Vectorizado (NumPy)  :     0.67 ms
Vectorizado frente a bucle       : 49 veces más rápido
Vectorizado frente a comprensión : 51 veces más rápido
```

#### Benchmark 2 — Suma de matrices 1000×1000

In [ ]:
tamano = 1000
lista_a = [[i + j for j in range(tamano)] for i in range(tamano)]
lista_b = [[i - j for j in range(tamano)] for i in range(tamano)]
arreglo_a = np.array(lista_a)
arreglo_b = np.array(lista_b)

def sumar_matrices_con_bucles(a, b):
    n_filas, n_columnas = len(a), len(a[0])
    resultado = [[0] * n_columnas for _ in range(n_filas)]
    for i in range(n_filas):
        for j in range(n_columnas):
            resultado[i][j] = a[i][j] + b[i][j]
    return resultado

tiempo_matriz_bucles = min(timeit.repeat(
    lambda: sumar_matrices_con_bucles(lista_a, lista_b), repeat=7, number=3
)) / 3

tiempo_matriz_vectorizada = min(timeit.repeat(
    lambda: arreglo_a + arreglo_b, repeat=7, number=10
)) / 10

razon = tiempo_matriz_bucles / tiempo_matriz_vectorizada
print(f"Bucles anidados : {tiempo_matriz_bucles * 1000:8.2f} ms")
print(f"Vectorizado     : {tiempo_matriz_vectorizada * 1000:8.2f} ms")
print(f"Vectorizado frente a bucles : {razon:.0f} veces más rápido")

*Salida en el libro:*

```text
Bucles anidados :    55.74 ms
Vectorizado     :     0.91 ms
Vectorizado frente a bucles : 61 veces más rápido
```

### Ejercicios propuestos

In [ ]:
import numpy as np

mediciones = np.array([10.2, 10.5, 10.1, 25.0, 10.3, 10.4])

# Se busca corregir un valor atípico evidente (25.0) SOLO en la copia de
# trabajo, sin tocar el array "mediciones" original.
subconjunto = mediciones[2:5]
subconjunto[1] = 10.35   # corrección del valor atípico

## Soluciones de los ejercicios

### [Ex-L2-C1-01] Vectorización de tres funciones del Libro 1

#### Desarrollo con código

In [ ]:
import numpy as np
import timeit

# ---------------------------------------------
# VERSIONES NATIVAS (las del Libro 1, sobre listas)
# ---------------------------------------------

def calcular_media_nativa(valores):
    """Media aritmética con acumulador y bucle explícito."""
    suma_total = 0.0
    for valor in valores:          # el bucle vive en el intérprete de Python
        suma_total += valor
    return suma_total / len(valores)


def normalizar_min_max_nativa(valores):
    """Normalización al rango [0, 1]: (x - min) / (max - min)."""
    valor_minimo = min(valores)    # primer recorrido completo de la lista
    valor_maximo = max(valores)    # segundo recorrido completo
    rango = valor_maximo - valor_minimo
    # tercer recorrido
    return [(valor - valor_minimo) / rango for valor in valores]


def aplicar_impuesto_nativa(precios, tasa_impuesto):
    """Aplica un impuesto porcentual a cada precio de la lista."""
    precios_finales = []
    for precio in precios:
        precios_finales.append(precio * (1 + tasa_impuesto))
    return precios_finales


# ---------------------------------------------
# VERSIONES VECTORIZADAS (sobre ndarray)
# ---------------------------------------------

def calcular_media_vectorizada(valores):
    """Media aritmética delegada a la agregación compilada de NumPy."""
    return valores.mean()


def normalizar_min_max_vectorizada(valores):
    """Normalización min-max sin bucles: la resta y la división son ufuncs."""
    valor_minimo = valores.min()
    return (valores - valor_minimo) / (valores.max() - valor_minimo)


def aplicar_impuesto_vectorizada(precios, tasa_impuesto):
    """El escalar se difunde (broadcasting) sobre todo el array."""
    return precios * (1 + tasa_impuesto)


# ---------------------------------------------
# PREPARACIÓN DE DATOS
# ---------------------------------------------
CANTIDAD_ELEMENTOS = 1_000_000
TASA_IMPUESTO = 0.16

# semilla fija: resultados reproducibles
generador = np.random.default_rng(42)
datos_array = generador.random(CANTIDAD_ELEMENTOS) * 1000
# misma información, estructura nativa
datos_lista = datos_array.tolist()

# PASO IMPRESCINDIBLE: comprobar que ambas versiones calculan LO MISMO
# antes de comparar cuál es más rápida. Una versión rápida e incorrecta no
# sirve.
print("misma media        :",
      np.isclose(calcular_media_nativa(datos_lista),
                 calcular_media_vectorizada(datos_array)))
print("misma normalización:",
      np.allclose(normalizar_min_max_nativa(datos_lista),
                  normalizar_min_max_vectorizada(datos_array)))
print("mismo impuesto     :",
      np.allclose(aplicar_impuesto_nativa(datos_lista, TASA_IMPUESTO),
                  aplicar_impuesto_vectorizada(datos_array, TASA_IMPUESTO)))


# ---------------------------------------------
# MEDICIÓN
# ---------------------------------------------
# En Jupyter o Google Colab basta la magia de celda:
#     %timeit calcular_media_nativa(datos_lista)
#     %timeit calcular_media_vectorizada(datos_array)
#
# Como script .py se usa el módulo timeit. Se toma el MÍNIMO de las
# repeticiones (no la media), siguiendo la metodología de la Sección 1.7.

def medir(funcion, repeticiones=5, ejecuciones=1):
    """Devuelve el tiempo mínimo por ejecución, en segundos."""
    return min(timeit.repeat(funcion, repeat=repeticiones,
                             number=ejecuciones)) / ejecuciones


tiempo_media_nativa = medir(lambda: calcular_media_nativa(datos_lista))
tiempo_media_vect = medir(lambda: calcular_media_vectorizada(datos_array),
                          7, 20)

tiempo_norm_nativa = medir(lambda: normalizar_min_max_nativa(datos_lista))
tiempo_norm_vect = medir(lambda: normalizar_min_max_vectorizada(datos_array),
                         7, 20)

tiempo_imp_nativa = medir(
    lambda: aplicar_impuesto_nativa(datos_lista, TASA_IMPUESTO))
tiempo_imp_vect = medir(
    lambda: aplicar_impuesto_vectorizada(datos_array, TASA_IMPUESTO), 7, 20)

print(f"\n{'Función':<24}{'Nativa (ms)':>12}{'Vectorizada (ms)':>18}"
      f"{'Aceleración':>13}")
for nombre, t_nativa, t_vect in [
        ("Media", tiempo_media_nativa, tiempo_media_vect),
        ("Normalización min-max", tiempo_norm_nativa, tiempo_norm_vect),
        ("Aplicación de impuesto", tiempo_imp_nativa, tiempo_imp_vect)]:
    print(f"{nombre:<24}{t_nativa * 1000:>12.2f}{t_vect * 1000:>18.3f}"
          f"{t_nativa / t_vect:>12.0f}x")

#### Salida esperada

*Salida en el libro:*

```text
misma media        : True
misma normalización: True
mismo impuesto     : True

Función                  Nativa (ms)  Vectorizada (ms)  Aceleración
Media                          22.53             0.351          64x
Normalización min-max          91.57             3.696          25x
Aplicación de impuesto         81.03             1.078          75x
```

### [Ex-L2-C1-02] Diagnóstico de vistas frente a copias

#### Desarrollo con código

In [ ]:
import numpy as np

mediciones_originales = np.array([10.2, 10.5, 10.1, 25.0, 10.3, 10.4])

# ---------------------------------------------
# A) EL CÓDIGO DEFECTUOSO
# ---------------------------------------------
mediciones = mediciones_originales.copy()

subconjunto = mediciones[2:5]     # SLICING BÁSICO -> devuelve una VISTA
subconjunto[1] = 10.35            # se escribe sobre la memoria COMPARTIDA

print(mediciones)
# [10.2  10.5  10.1  10.35 10.3  10.4 ]   <- el 25.0 original desapareció

# La evidencia objetiva del diagnóstico: .base revela el origen de la memoria.
print(subconjunto.base is mediciones)   # True -> comparten el mismo bloque


# ---------------------------------------------
# B) LA CORRECCIÓN: .copy() EXPLÍCITO
# ---------------------------------------------
mediciones = mediciones_originales.copy()

subconjunto_corregido = mediciones[2:5].copy()   # copia real e independiente
subconjunto_corregido[1] = 10.35

print(mediciones)               # [10.2 10.5 10.1 25.  10.3 10.4]  <- intacto
print(subconjunto_corregido)    # [10.1  10.35 10.3 ]
print(subconjunto_corregido.base)   # None -> no depende de memoria ajena


# ---------------------------------------------
# C) CONTRASTE: INDEXACIÓN AVANZADA (FANCY INDEXING)
# ---------------------------------------------
mediciones = mediciones_originales.copy()

# lista de índices -> COPIA automática
subconjunto_fancy = mediciones[[2, 3, 4]]
subconjunto_fancy[1] = 10.35

print(mediciones)
# [10.2 10.5 10.1 25.  10.3 10.4]  <- intacto
print(subconjunto_fancy.base)      # None


# ---------------------------------------------
# D) LAS MÁSCARAS BOOLEANAS TAMBIÉN COPIAN
# ---------------------------------------------
mediciones = mediciones_originales.copy()

mascara = mediciones > 20
sub_mascara = mediciones[mascara]     # indexación booleana -> COPIA
sub_mascara[0] = 0.0

print(mediciones)              # [10.2 10.5 10.1 25.  10.3 10.4]  <- intacto
print(sub_mascara.base)        # None

In [ ]:
def probar(seleccionar):
    """Selecciona, modifica la selección y comprueba el efecto."""
    mediciones = mediciones_originales.copy()
    seleccion = seleccionar(mediciones)
    seleccion[0] = 0.0
    alterado = not np.array_equal(mediciones, mediciones_originales)
    return seleccion.base is mediciones, alterado


casos = [("A) mediciones[2:5]", lambda m: m[2:5]),
         ("B) mediciones[2:5].copy()", lambda m: m[2:5].copy()),
         ("C) mediciones[[2, 3, 4]]", lambda m: m[[2, 3, 4]]),
         ("D) mediciones[mascara]", lambda m: m[m > 20])]

print(f"{'Caso':<28}{'.base is mediciones':>21}{'¿Alteró el original?':>22}")
for nombre, seleccionar in casos:
    comparte, alterado = probar(seleccionar)
    print(f"{nombre:<28}{comparte!s:>21}{alterado!s:>22}")

#### Salida esperada

*Salida en el libro:*

```text
Caso                          .base is mediciones  ¿Alteró el original?
A) mediciones[2:5]                           True                  True
B) mediciones[2:5].copy()                   False                 False
C) mediciones[[2, 3, 4]]                    False                 False
D) mediciones[mascara]                      False                 False
```